In [20]:
two = 2
three = 3
two*three

6

In [21]:
import msprime
import dadi
import numpy as np

In [22]:
demography = msprime.Demography()

In [23]:
demography.add_population(name="pop", initial_size=500)  # present size

# ancestral size before T=800 generations ago was 10,000
demography.add_population_parameters_change(
    time=800, initial_size=10_000, population="pop"
)

PopulationParametersChange(time=800, initial_size=10000, growth_rate=None, population='pop')

In [24]:
demography

Demography(populations=[Population(initial_size=500, growth_rate=0, name='pop', description='', extra_metadata={}, default_sampling_time=None, initially_active=None, id=0)], events=[PopulationParametersChange(time=800, initial_size=10000, growth_rate=None, population='pop')], migration_matrix=array([[0.]]))

In [25]:
ts = msprime.sim_ancestry(
    samples={"pop": 20},
    demography=demography,
    sequence_length=5_000_000,
    recombination_rate=1e-8,
    random_seed=42,
)

In [33]:
mts = msprime.sim_mutations(ts, rate=1e-8, random_seed=43)

In [70]:
mts.genotype_matrix()
with open("bottleneck_sim.vcf", "w") as f:
    mts.write_vcf(f)

In [71]:
vcf_file = "bottleneck_sim.vcf"
popfile = "pops.txt"
dd = dadi.Misc.make_data_dict_vcf(vcf_file, popfile)

In [72]:
fs = dadi.Spectrum.from_data_dict(
    dd,
    pop_ids=["pop"],
    projections=[30],
    polarized=False,   # folded SFS
)
print("Spectrum sample size:", fs.sample_sizes)
print("Segregating sites:", fs.S())

Spectrum sample size: [30]
Segregating sites: 2844.6406640560062


In [73]:
def snm(params, ns, pts): # define single population model, no free parameters
    xx = dadi.Numerics.default_grid(pts)
    phi = dadi.PhiManip.phi_1D(xx)
    fs_model = dadi.Spectrum.from_phi(phi, ns, (xx,))
    return fs_model

def two_epoch(params, ns, pts): # define bottlenneck model
    nu, T = params # two free parameters: scaled current pop size and time of split 
    xx = dadi.Numerics.default_grid(pts)
    phi = dadi.PhiManip.phi_1D(xx)
    phi = dadi.Integration.one_pop(phi, xx, T, nu)
    fs_model = dadi.Spectrum.from_phi(phi, ns, (xx,))
    return fs_model

In [74]:
pts_l = [40, 50, 60]
snm_ex = dadi.Numerics.make_extrap_log_func(snm)
two_epoch_ex = dadi.Numerics.make_extrap_log_func(two_epoch)

In [75]:
model_snm = snm_ex([], fs.sample_sizes, pts_l)
theta_snm = dadi.Inference.optimal_sfs_scaling(model_snm, fs)
ll_snm = dadi.Inference.ll_multinom(model_snm, fs)

print("\nConstant-size model")
print("log-likelihood:", ll_snm)
print("theta:", theta_snm)


Constant-size model
log-likelihood: -430.50272646957603
theta: 719.989428931301


In [76]:
p0 = [0.5, 0.1]  
lower_bound = [1e-3, 1e-4]
upper_bound = [20, 10]
p0_perturbed = dadi.Misc.perturb_params(
    p0, fold=1, lower_bound=lower_bound, upper_bound=upper_bound
)

In [77]:
popt = dadi.Inference.optimize_log(
    p0_perturbed,
    fs,
    two_epoch_ex,
    pts_l,
    lower_bound=lower_bound,
    upper_bound=upper_bound,
    verbose=1,
    maxiter=50,
)

1       , -427.403    , array([ 0.990463   ,  0.119143   ])
2       , -427.726    , array([ 0.991454   ,  0.119143   ])
3       , -427.401    , array([ 0.990463   ,  0.119263   ])
4       , -160.39     , array([ 0.360748   ,  0.119639   ])
5       , -160.561    , array([ 0.361109   ,  0.119639   ])
6       , -160.363    , array([ 0.360748   ,  0.119759   ])
10      , -160.389    , array([ 0.360748   ,  0.11964    ])
11      , -160.56     , array([ 0.361109   ,  0.11964    ])
12      , -160.362    , array([ 0.360748   ,  0.11976    ])
13      , -160.384    , array([ 0.360742   ,  0.119651   ])
14      , -160.555    , array([ 0.361103   ,  0.119651   ])
15      , -160.357    , array([ 0.360742   ,  0.119771   ])
19      , -160.384    , array([ 0.360742   ,  0.119651   ])
20      , -160.555    , array([ 0.361103   ,  0.119651   ])
21      , -160.357    , array([ 0.360742   ,  0.119771   ])
22      , -160.384    , array([ 0.360742   ,  0.119651   ])
23      , -160.555    , array([ 0.361103

In [78]:
model_two = two_epoch_ex(popt, fs.sample_sizes, pts_l)
theta_two = dadi.Inference.optimal_sfs_scaling(model_two, fs)
ll_two = dadi.Inference.ll_multinom(model_two, fs)

print("\nTwo-epoch model")
print("best params [nu, T]:", popt)
print("log-likelihood:", ll_two)
print("theta:", theta_two)


Two-epoch model
best params [nu, T]: [0.02649488 0.0539727 ]
log-likelihood: -70.74075923060917
theta: 6155.471158862881


In [79]:
print("\nModel comparison")
print(f"Delta log-likelihood (two-epoch - constant): {ll_two - ll_snm:.3f}")

# constant model has k=0 free params in this formulation
# two_epoch has k=2
aic_snm = 2 * 0 - 2 * ll_snm
aic_two = 2 * 2 - 2 * ll_two

print(f"AIC constant: {aic_snm:.3f}")
print(f"AIC two-epoch: {aic_two:.3f}")


Model comparison
Delta log-likelihood (two-epoch - constant): 359.762
AIC constant: 861.005
AIC two-epoch: 145.482


In [80]:
print("True parameters:")
print("nu =", 0.05)
print("T  =", 0.04)

print("\nInferred parameters:")
print("nu =", popt[0])
print("T  =", popt[1])

True parameters:
nu = 0.05
T  = 0.04

Inferred parameters:
nu = 0.026494883113851595
T  = 0.0539726968369783


In [81]:
N_anc = 10_000  # known from simulation
nu_est, T_est = popt # label popt parameter estimates
N_curr_est = nu_est * N_anc # multiply Nu in dadi units by ancestral populaiton size to get current N_e
t_est = T_est * 2 * N_anc # multiply T in dadi units by 2N_e to get generations
print("Estimated current size:", N_curr_est)
print("Estimated bottleneck time (generations):", t_est)

Estimated current size: 264.94883113851597
Estimated bottleneck time (generations): 1079.453936739566


The infered parameter values are closer between the T paramaters than they are for the nu parameters, the model estimated that the current population size was around 265 individuals which is about half of what the population we created was. The generations since bottleneck was 1080 which is a closer estimate than the number of individuals, we set the generations since bottleneck to 800 and the model was less than 300 generations of. The AIC was extremely high for these models suggesting that there wasnt great fit which could be why the estimations on population size and bottleneck time were so far off. 